# Refine LLM labelling

In [1]:
from discovery_utils.utils import google
from src import utils as src_utils
from src import PROJECT_DIR

import pandas as pd

In [2]:
from discovery_utils.utils.llm import batch_check

PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}/llm_labelling"


In [3]:
sheet_id = "1m9_tKyJDaSy2vDWxYVP_9HlfBbGysQUV-xrb1FW3vok"
tab_name = "ukri_check"

In [4]:
CONFIG_NAMES = [
    "bioenergy",
    "biomass_heating",
    "built_environment",
    "ccus",
    "district_heating",
    "energy_efficiency",
    "energy_grid",
    "geothermal_energy",
    "green_skills",
    "heat_pumps",
    "heat_storage",
    "hydrogen_energy",
    "hydrogen_heating",
    "micro_chp",
    "solar_thermal",
    "energy_storage",    
    "solar",
    "wind"
]

In [5]:
checked_gtr_df = google.access_google_sheet(sheet_id, "ukri_check")

In [6]:
checked_cb_df = google.access_google_sheet(sheet_id, "crunchbase_check")

In [267]:
category = "heat_pumps"
config_dict = src_utils.get_config_dict(category)
theme = config_dict["search_recipe"]["category_name"]

In [268]:
# combine values in "reviewer (karlis)" and reviewer (will)" columns in one column
gtr_data_df = (
    checked_gtr_df
    # replace empty strings with NaN
    .replace("", pd.NA)
    .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    .query("theme == @category")
    .assign(dataset="gtr")
)

cb_data_df = (
    checked_cb_df
    # .assign(reviewer = lambda df: df["reviewer (karlis)"].combine_first(df["reviewer (will)"]))
    .assign(reviewer = lambda df: df["reviewer (karlis)"])
    .query("theme == @category")
    .assign(dataset="crunchbase")
)

data_df = pd.concat([gtr_data_df, cb_data_df], ignore_index=True).dropna(subset=["reviewer"])
human_data_df = data_df[["theme", "id", "text", "reviewer", "dataset"]]
llm_data_df = data_df[["id", "is_relevant"]].copy()

In [269]:
human_data_df

,theme,id,text,reviewer,dataset
0,heat_pumps,26B5EABB-1D7F-4EB7-85F4-96A73408D098,Addressing the complexity of future power syst...,no,gtr
1,heat_pumps,EDAD88DF-6519-4EFA-A9B8-498E165F1AB5,"Built environment, energy, digital and transpo...",yes,gtr
2,heat_pumps,5901A18E-926B-43A3-8DC7-3BF623D07017,End Use Energy Demand Centres Collaborative Pr...,yes,gtr
3,heat_pumps,7AF9D240-72D6-4011-8FE5-766F885F52B6,GasNetNew - The role of the gas network in a f...,yes,gtr
4,heat_pumps,43B4EF23-814F-4C3A-8A73-68A029809978,"homeBRU, the development of a 3kWe microCHP un...",no,gtr
5,heat_pumps,DA086D67-7253-4378-9394-087EE6BBAEBC,Innovate Composite Smart Materials for Heating...,yes,gtr
6,heat_pumps,77A9466B-34A4-4C01-BF42-F473E09BABBF,Milford Haven: Energy Kingdom &quot;A national...,yes,gtr
7,heat_pumps,5ECAA0BF-FF45-4D7D-A387-1C7FFF50C32B,Smart Energy Research Lab The UK is investing ...,no,gtr
8,heat_pumps,E349FB40-14A3-4B99-962E-4800FF821B83,Topology Optimization for Additive manufacturi...,no,gtr
9,heat_pumps,DD5D797B-74BA-4D69-BA80-47BEF3FD12A0,V2BUILD The current energy crisis has increase...,yes,gtr


In [270]:
compare_df = (
    human_data_df
    .merge(llm_data_df, on="id", how="left")
    .assign(agreement = lambda df: df["is_relevant"] == df["reviewer"])
)
# total accuracy metric
total_acc = compare_df.agreement.mean()
# group by dataset
dataset_acc = (
    compare_df
    .groupby("dataset")
    .agg(total_acc = ("agreement", "mean"))
    .reset_index()
)
# check a suite of accuracy metrics, FP, FN, recall, precision, F1
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

def metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True)
    accuracy = report["accuracy"]
    precision = report["1"]["precision"]
    recall = report["1"]["recall"]
    f1_score = report["1"]["f1-score"]
    return accuracy, precision, recall, f1_score

def plot_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.title('Confusion Matrix')
    plt.show()

def get_reports(compare_df):
    reports = dict()
    reports['total'] = classification_report(
        compare_df["reviewer"],
        compare_df["is_relevant"],
        output_dict=True
    )
    for dataset in compare_df["dataset"].unique():
        reports[dataset] = classification_report(
            compare_df.loc[compare_df["dataset"] == dataset, "reviewer"],
            compare_df.loc[compare_df["dataset"] == dataset, "is_relevant"],
            output_dict=True
        )
    return reports
    # plot_confusion_matrix(compare_df["reviewer"], compare_df["is_relevant"], labels=["yes", "no"])
original_report = get_reports(compare_df)

In [271]:
compare_df.groupby(["dataset", "reviewer"]).agg(total_acc = ("agreement", "mean"), support=("id", "count")).reset_index()

,dataset,reviewer,total_acc,support
0,crunchbase,no,0.7500,4
1,crunchbase,yes,0.5625,16
2,gtr,no,1.0000,4
3,gtr,yes,0.6250,16


In [272]:
compare_df.agreement.mean()

0.65

In [273]:
original_report['total']

{'no': {'precision': 0.35, 'recall': 0.875, 'f1-score': 0.5, 'support': 8.0},
 'yes': {'precision': 0.95,
  'recall': 0.59375,
  'f1-score': 0.7307692307692307,
  'support': 32.0},
 'accuracy': 0.65,
 'macro avg': {'precision': 0.6499999999999999,
  'recall': 0.734375,
  'f1-score': 0.6153846153846154,
  'support': 40.0},
 'weighted avg': {'precision': 0.8299999999999998,
  'recall': 0.65,
  'f1-score': 0.6846153846153846,
  'support': 40.0}}

In [235]:
check_data = dict(zip(data_df['id'], data_df['text']))

system_message = batch_check.generate_relevance_check_system_message(config_dict)
system_message += """ 
Mark the text as 'yes' if (one or more of the following):
- If the technology defined by the scope above, is the main focus
- If the technology is one of the components or activities described by the text. For example the technology could be  mentioned
    as part of a larger project or business including other technologies, or be mentioned as one of the use cases or case studies.
- If the text describes a company and the technology is the main focus, or a part of a broader range of the company's activities and offerings.
- If the text is about a component or critical element of the technology defined above
- If the text describes a technology or process and explicitly mentions that it can be applied on the technology defined above to improve it's performance or efficiency.

If the text is about heating technology but application target is not mentioned then assume it could be relevant for households or buildings (as opposed to an industrial applications).

However, mark it as 'no' if (one or more of the following):
- The activities, or business described in the text does not have a discernable impact on or connection with the technology.
- If the technology is mentioned only in passing or as a minor example in a broader discussion, for example, 
    in only one sentence within a long text with many sentences, or at the very end of a long description.
- The technology is mentioned only as a negative example (eg "unlike [technology]...")
- The text mentions heat pumps for heating swimming pools  
- The text would be better captured by one of the other categories (comma separated) mentioned in this list:  
Bioenergy (biofuels), Biomass heating, Carbon capture and storage, District heating and heat networks, Energy grid, Geothermal energy, 
Heat pumps, Hydrogen energy, Hydrogen heating, Micro CHP, Solar thermal heating, Energy storage (batteries), Solar power, Wind power
"""    
        
fields = [
    {"name": "explanation", "type": "str", "description": "A short, 1-sentence explanation of the answer."},
    {"name": "is_relevant", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},    
]

In [236]:
print(system_message)

Determine whether the text provided by the user is relevant based on the defined scope.

The scope of this task includes the following:
- Sustainable and energy-efficient heat pump solutions for residential heating
- Decarbonising home heating through heat pump technology
- Integration of heat pumps with smart home and renewable energy systems

The relevant texts will usually combine these or highly similar keywords from each of the following sets:
- Heat pumps keywords: heat pumps, heat pump


Use the sentences and keywords given above, to determine if the text provided by the user is relevant. Note that the lists of keywords and sentences are not exhaustive, but provide the main criteria for relevance. Determine also other information from the text as defined below. 
Mark the text as 'yes' if (one or more of the following):
- If the technology defined by the scope above, is the main focus
- If the technology is one of the components or activities described by the text. For example th

In [237]:
file_name = str(OUTPUT_DIR / f"llm_refinement_gtr_{category}.jsonl")
# if file_name exists, delete it
import os
if os.path.exists(file_name):
    os.remove(file_name)

processor = batch_check.LLMProcessor(
    # model_name="o3-mini",
    # temperature=None,
    model_name="gpt-4o-mini",
    temperature=0.0,
    output_path=file_name,
    system_message=system_message,
    session_name="mission_studio",
    output_fields=fields,

)

await processor.run(check_data, batch_size=10, sleep_time=0.3)    

2025-04-11 16:55:32,495 - root - INFO - Using OpenAI
2025-04-11 16:55:32,533 - root - INFO - Processing batch 1/4
2025-04-11 16:55:34,612 - root - INFO - Processing batch 2/4
2025-04-11 16:55:37,106 - root - INFO - Processing batch 3/4
2025-04-11 16:55:39,251 - root - INFO - Processing batch 4/4


In [238]:
compare_df = (
    human_data_df
    .merge(pd.read_json(file_name, lines=True), how="left", on="id")
    .assign(agreement = lambda df: df["is_relevant"] == df["reviewer"])
)

In [239]:
compare_df.sort_values(["dataset", "reviewer", "is_relevant"])

,theme,id,text,reviewer,dataset,explanation,is_relevant,timestamp,model,temperature,agreement
22,heat_pumps,451db563-6a7f-4341-ba91-5b7291d5bdd1,G&F Manufacturing G&F Manufacturing produces G...,no,crunchbase,The text focuses specifically on pool heat pum...,no,2025-04-11 15:55:37.110261+00:00,gpt-4o-mini,0,True
26,heat_pumps,f02f1271-e3a9-4605-a9ee-8a7c5324018f,PROCOPI PROCOPI is a multi-specialist supplier...,no,crunchbase,The text primarily focuses on pool and spa equ...,no,2025-04-11 15:55:37.110942+00:00,gpt-4o-mini,0,True
32,heat_pumps,468eabec-880c-468b-a76d-1d5063013fc2,Aqualazer Aqualazer produces vinyl coatings fo...,no,crunchbase,The text mentions heat pumps as part of a broa...,no,2025-04-11 15:55:39.255504+00:00,gpt-4o-mini,0,True
28,heat_pumps,46cc648b-50f5-45b3-ae62-fc1fb61136f2,The Energy Warrior The Energy Warrior provides...,no,crunchbase,The text focuses on heat pump hot water soluti...,yes,2025-04-11 15:55:37.111259+00:00,gpt-4o-mini,0,False
23,heat_pumps,f6bde1d3-b181-4eda-bb35-248c661ca40b,KwiKool KwiKool is a manufacturer of portable ...,yes,crunchbase,The text mentions heat pumps as part of a broa...,no,2025-04-11 15:55:37.110446+00:00,gpt-4o-mini,0,False
27,heat_pumps,fae1edde-1948-4be8-80b0-a73451f8f751,Sunny Service Sunny Service is an air conditio...,yes,crunchbase,The text describes a company that provides HVA...,no,2025-04-11 15:55:37.111102+00:00,gpt-4o-mini,0,False
20,heat_pumps,c2c7e7d1-e89d-4835-bf65-a1e3509432f1,Cook's Comfort Systems Cook's Comfort Systems ...,yes,crunchbase,The text mentions heat pump service as part of...,yes,2025-04-11 15:55:37.109488+00:00,gpt-4o-mini,0,True
21,heat_pumps,c4528709-742f-4c85-83ac-f85a73037895,EnergySol EnergySol is a suppliers of solar sy...,yes,crunchbase,The text mentions heat pumps as part of a broa...,yes,2025-04-11 15:55:37.110032+00:00,gpt-4o-mini,0,True
24,heat_pumps,50c5c3e7-44b2-4209-bc6b-0b6b19bd34db,Mini Split Warehouse Mini Split Warehouse sell...,yes,crunchbase,The text primarily focuses on mini-split heat ...,yes,2025-04-11 15:55:37.110615+00:00,gpt-4o-mini,0,True
25,heat_pumps,53933e10-2faa-44d3-8bb6-9f485de58065,"Modern Air Modern Air repairs, installs, and m...",yes,crunchbase,The text mentions heat pumps as part of the HV...,yes,2025-04-11 15:55:37.110780+00:00,gpt-4o-mini,0,True


In [247]:
results_df = compare_df.groupby(["dataset", "reviewer"]).agg(total_acc = ("agreement", "mean"), support=("id", "count")).reset_index()
print(results_df)

      dataset reviewer  total_acc  support
0  crunchbase       no      0.750        4
1  crunchbase      yes      0.875       16
2         gtr       no      0.500        4
3         gtr      yes      1.000       16


In [248]:
print(compare_df.agreement.mean())

0.875


In [ ]:
reports = get_reports(compare_df)


In [246]:
reports

{'total': {'no': {'precision': 0.7142857142857143,
   'recall': 0.625,
   'f1-score': 0.6666666666666666,
   'support': 8.0},
  'yes': {'precision': 0.9090909090909091,
   'recall': 0.9375,
   'f1-score': 0.9230769230769231,
   'support': 32.0},
  'accuracy': 0.875,
  'macro avg': {'precision': 0.8116883116883117,
   'recall': 0.78125,
   'f1-score': 0.7948717948717949,
   'support': 40.0},
  'weighted avg': {'precision': 0.8701298701298701,
   'recall': 0.875,
   'f1-score': 0.8717948717948719,
   'support': 40.0}},
 'gtr': {'no': {'precision': 1.0,
   'recall': 0.5,
   'f1-score': 0.6666666666666666,
   'support': 4.0},
  'yes': {'precision': 0.8888888888888888,
   'recall': 1.0,
   'f1-score': 0.9411764705882353,
   'support': 16.0},
  'accuracy': 0.9,
  'macro avg': {'precision': 0.9444444444444444,
   'recall': 0.75,
   'f1-score': 0.803921568627451,
   'support': 20.0},
  'weighted avg': {'precision': 0.9111111111111111,
   'recall': 0.9,
   'f1-score': 0.8862745098039216,
   'su